# M10.5 · Paralel indirgeme ve histogram — ağaçlar, ayrışmasız eşleme, karıştırma, özelleştirme

İndirgeme: birleşmeli (gerekirse değişmeli) bir işleç ve birim elemanla bir diziyi tek değere indirmek (toplam, max, argmax, çarpım). Sıralı O(N), indirgeme ağacı O(log N) adım ve N − 1 iş; CUDA'da basit eşleme (adım ikiye katlanır, sahip thread'ler dağınık → ayrışma) ile yakınsak eşleme (adım yarıya iner, aktif thread'ler bitişik), paylaşımlı bellek, blok hiyerarşisi + atomik birleştirme, thread kabalaştırma (coarsening), warp karıştırma (`__shfl_xor_sync`). Histogram: global atomikler, paylaşımlı bellekte özelleştirme, çakışma ve birleştirme. Kayan nokta toplamının birleşmeli olmaması: sıralı, ikili (pairwise) ve Kahan toplamı. Saf Python'da warp etkinliği sayımı, NumPy'da ağaç indirgemeleri, Triton'da iki geçişli/atomik toplam, argmax ve histogram, C++'ta çoklu biriktirici ve çoklu alt-histogram.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-5-reduction.html](https://m-gonul.github.io/pytorch-lab/m10-5-reduction.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_5_reduction.py
CHECKS = {
 "b": "import random\nfrom labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def tree():\n        ensure(need(ns, \"reduce_tree\")([3, 1, 7, 0, 4, 1, 6, 3], max, float(\"-inf\")) == (7, 3, 7), \"el örneği\")\n        r = ns[\"reduce_tree\"](list(range(1, 1001)), lambda a, b: a + b, 0)\n        ensure(r == (500500, 10, 999), \"1000 eleman: 10 adım, 999 işlem\")\n        cat = ns[\"reduce_tree\"](list(\"paralel\"), lambda a, b: a + b, \"\")\n        ensure(cat[0] == \"paralel\", \"birleşmeli ama değişmeli olmayan işleç: sıra korunmalı\")\n    R_.test(\"reduce_tree\", tree)\n\n    def warps():\n        s = need(ns, \"warp_activity\")(256, \"simple\")\n        c = ns[\"warp_activity\"](256, \"convergent\")\n        ensure(sum(w for _, w, _ in s) == 55 and sum(w for _, w, _ in c) == 20, \"etkin warp toplamları 55 ve 20\")\n        ensure(sum(d for *_, d in s) == 8 * 5 + 4 + 2 + 1 and sum(d for *_, d in c) == 5, \"ayrışan warp sayıları\")\n        rng = random.Random(0)\n        x = [rng.randint(0, 9) for _ in range(512)]\n        ensure(need(ns, \"run_block\")(x, \"simple\") == sum(x) and ns[\"run_block\"](x, \"convergent\") == sum(x), \"iki eşleme de doğru\")\n        rows = [[i + 1, f\"{s[i][0]} / {s[i][1]} / {s[i][2]}\", f\"{c[i][0]} / {c[i][1]} / {c[i][2]}\"] for i in range(9)]\n        R_.table(\"256 thread: aktif thread / aktif warp / ayrışan warp\", [\"iterasyon\", \"basit (adım ×2)\", \"yakınsak (adım ÷2)\"], rows,\n                 note=\"Aynı toplama sayısı, çok farklı donanım kullanımı: basit eşlemede aktif warp her zaman 'yarı boş'.\")\n        data = [rng.randint(0, 15) for _ in range(10000)]\n        h, m = need(ns, \"hist_private\")(data, 8, 16)\n        ensure(h == [data.count(b) for b in range(16)] and m == 8 * 16, \"özel histogram\")\n    R_.test(\"warp etkinliği ve histogram\", warps)\n",
 "c": "import math\nimport numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 1, func=\"tree_reduce\"), max_loops(src, 1, func=\"argmax_tree\"), max_loops(src, 1, func=\"chain_product_tree\")))\n    rng = np.random.default_rng(0)\n\n    def t():\n        close(need(ns, \"tree_reduce\")(np.arange(1, 1001), np.add, 0), 500500, what=\"toplam\")\n        x = rng.normal(size=777)\n        close(ns[\"tree_reduce\"](x, np.maximum, -np.inf), x.max(), what=\"max\")\n        close(ns[\"tree_reduce\"](x[:5], np.minimum, np.inf), x[:5].min(), what=\"tek uzunluk\")\n        ensure(need(ns, \"argmax_tree\")(x) == int(np.argmax(x)), \"argmax\")\n        ensure(ns[\"argmax_tree\"](np.array([1, 5, 5, 2, 5])) == 1, \"eşitlikte ilk indeks\")\n        M = rng.normal(size=(37, 2, 2))\n        ref = np.eye(2)\n        for m in M:\n            ref = ref @ m\n        close(need(ns, \"chain_product_tree\")(M), ref, rtol=1e-9, what=\"sıralı matris çarpımı = ağaç\")\n        rows = []\n        for n in (10 ** 4, 10 ** 6, 10 ** 7):\n            v = rng.random(n).astype(np.float32)\n            ref = math.fsum(v.astype(np.float64))\n            a, b = need(ns, \"naive_sum32\")(v), need(ns, \"tree_sum32\")(v)\n            rows.append([n, f\"{abs(a - ref) / ref:.1e}\", f\"{abs(b - ref) / ref:.1e}\"])\n        ensure(float(rows[-1][2]) < float(rows[-1][1]) / 10, \"ağaç toplamı çok daha doğru\")\n        R_.table(\"float32 toplamın göreli hatası (U[0,1) sayılar)\", [\"n\", \"sıralı\", \"ağaç (ikili)\"], rows,\n                 note=\"Sıralıda biriktirici büyüdükçe her yeni küçük sayı daha çok yuvarlanır (hata ~nε). Ağaçta her toplama benzer büyüklükleri toplar (~ε log n). GPU indirgemesi böylece 'yan etki olarak' daha doğrudur.\")\n    R_.test(\"ağaç indirgemeleri\", t)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    dev = ns.get(\"DEVICE\", \"cpu\")\n    g = torch.Generator().manual_seed(0)\n\n    def sums():\n        for n in (1, 1000, 1024, 5000, 300000):\n            x = torch.randn(n, generator=g).to(dev)\n            ref = float(x.double().sum())\n            close(float(need(ns, \"sum_two_pass\")(x)), ref, atol=1e-3 * max(1, abs(ref)) + 1e-3, what=f\"iki geçiş n = {n}\")\n            close(float(need(ns, \"sum_atomic\")(x)), ref, atol=1e-3 * max(1, abs(ref)) + 1e-3, what=f\"atomik n = {n}\")\n            ensure(need(ns, \"argmax\")(x) == int(torch.argmax(x)), f\"argmax n = {n}\")\n        ensure(need(ns, \"block_sums\")(torch.ones(3000).to(dev)).tolist() == [1024.0, 1024.0, 952.0], \"blok toplamları\")\n    R_.test(\"toplam ve argmax\", sums)\n\n    def hist():\n        x = torch.randint(0, 50, (20000,), generator=g, dtype=torch.int32).to(dev)\n        ensure(torch.equal(need(ns, \"histogram\")(x, 50).cpu(), torch.bincount(x.cpu(), minlength=50).int()), \"histogram\")\n        y = torch.full((5000,), 7, dtype=torch.int32).to(dev)\n        ensure(int(ns[\"histogram\"](y, 10)[7]) == 5000, \"tek değerli girdi: atomikler hiçbirini kaybetmemeli\")\n    R_.test(\"histogram\", hist)\n\n    def bf():\n        x = torch.randn(10, 32, generator=g, dtype=torch.float64)\n        out = need(ns, \"butterfly\")(x)\n        close(out.numpy(), x.sum(1, keepdim=True).expand(10, 32).numpy(), atol=1e-12, what=\"her şerit warp toplamını tutar\")\n        R_.note(\"Kelebek: 5 adımda her şerit (lane) toplamı görür — paylaşımlı bellek ve __syncthreads olmadan. __shfl_down_sync ise yalnız şerit 0'da sonuç bırakır.\")\n    R_.test(\"butterfly (__shfl_xor_sync)\", bf)\n",
 "e": "import os\nimport torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    ext = need(ns, \"build\")()\n    P = max(2, min(8, os.cpu_count() or 2))\n    g = torch.Generator().manual_seed(0)\n\n    def s():\n        x = torch.rand(20_000_000, generator=g)\n        ref = float(x.double().sum())\n        rows = []\n        for k in (1, 2, 4, 8):\n            v = ext.sum_acc(x, 1, k)\n            close(v, ref, rtol=1e-3, what=f\"k = {k}\")\n            rows.append([f\"1 thread, {k} biriktirici\", fmt_time(bench(lambda: ext.sum_acc(x, 1, k), repeat=3, number=1)), f\"{abs(v - ref) / ref:.1e}\"])\n        rows.append([f\"{P} thread, 8 biriktirici\", fmt_time(bench(lambda: ext.sum_acc(x, P, 8), repeat=3, number=1)), f\"{abs(ext.sum_acc(x, P, 8) - ref) / ref:.1e}\"])\n        rows.append([\"torch.sum (float32)\", fmt_time(bench(lambda: x.sum(), repeat=3, number=1)), f\"{abs(float(x.sum()) - ref) / ref:.1e}\"])\n        R_.table(\"20 M float32 toplamı\", [\"yöntem\", \"süre\", \"göreli hata\"], rows,\n                 note=\"Tek biriktiricide her toplama öncekinin sonucunu bekler (FP toplama gecikmesi ~4 çevrim): döngü gecikme-bağımlı. k biriktirici k bağımsız zincir = CPU'da 'ağaç'ın ilk seviyesi; hem hızlı hem daha doğru. -O2 kayan nokta toplamını kendisi yeniden sıralamaz (birleşmeli değil!).\")\n    R_.test(\"çoklu biriktirici\", s)\n\n    def h():\n        img = torch.randint(0, 256, (4096, 4096), dtype=torch.uint8, generator=g)\n        flat = torch.full((4096, 4096), 42, dtype=torch.uint8)\n        rows = []\n        for name, im in ((\"tekdüze rastgele\", img), (\"tek renk\", flat)):\n            ref = torch.bincount(im.flatten(), minlength=256)\n            for c in (1, 4):\n                ensure(torch.equal(ext.hist_copies(im, P, c), ref), f\"{name}, {c} kopya\")\n                rows.append([name, c, fmt_time(bench(lambda: ext.hist_copies(im, P, c), repeat=3, number=1))])\n        R_.table(f\"16 M piksel histogramı, {P} thread (özel histogramlar)\", [\"görüntü\", \"thread başına alt-histogram\", \"süre\"], rows,\n                 note=\"Tek renkte her piksel aynı sayaca yazar: yükle–artır–yaz zinciri bir öncekinin bitmesini bekler (CPU'daki 'atomik çakışma' benzeri). 4 kopya zinciri 4'e böler. GPU'da aynı fikir: warp başına alt-histogram.\")\n    R_.test(\"çoklu alt-histogram\", h)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for n in (10 ** 3, 10 ** 5, 10 ** 6):\n            e = need(ns, \"errors\")(n, 1)\n            rows.append([n] + [f\"{v:.1e}\" for v in e])\n        last = [float(v) for v in rows[-1][1:]]\n        ensure(last[2] < 1e-6 and last[2] < last[0] / 100, \"Kahan çok daha doğru\")\n        ensure(float(need(ns, \"kahan_sum\")([1.0] + [1e-8] * 1000)) > 1.0, \"Kahan küçük terimleri kaybetmez\")\n        R_.table(\"float32 toplamın göreli hatası\", [\"n\", \"sıralı\", \"ikili (ağaç)\", \"Kahan\", \"np.sum\"], rows,\n                 note=\"np.sum zaten 8'li bloklarla ikili toplama yapar. Kahan her adımda kaybolan alçak bitleri c'de taşır; bedeli 4 kat FP işlem ve derleyicinin 'akıllı' cebirsel sadeleştirmesine kapalı olmak (-ffast-math Kahan'ı bozar).\")\n    R_.test(\"toplam hassasiyeti\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"warp_serialization\"))\n\n    def t():\n        close(need(ns, \"warp_serialization\")(np.stack([np.zeros(32), np.arange(32), np.arange(32) % 4])), np.array([32, 1, 8]), what=\"warp başına sıralama\")\n        rng = np.random.default_rng(0)\n        imgs = {\"tekdüze\": rng.integers(0, 256, 65536), \"doğal benzeri (N(128, 20))\": np.clip(rng.normal(128, 20, 65536), 0, 255).astype(int),\n                \"düzgün bölgeli (32'lik koşular)\": np.repeat(rng.integers(0, 256, 2048), 32), \"tek renk\": np.full(65536, 42)}\n        rows = []\n        for name, im in imgs.items():\n            g, _ = need(ns, \"cost\")(im, 256, False)\n            sh, merge = ns[\"cost\"](im, 256, True, 256)\n            rows.append([name, g, f\"{g / (65536 // 32):.1f}\", f\"{sh} + {merge}\"])\n        ensure(rows[3][1] == 65536 and rows[0][1] < 3 * 65536 // 32, \"tek renkte tam sıralama\")\n        R_.table(\"65 536 piksel, warp başına atomik sıralama adımları\", [\"görüntü\", \"global atomik adımı\", \"warp başına\", \"özel: blok içi adım + global birleştirme\"], rows,\n                 note=\"Doğal görüntülerde komşu pikseller benzer: warp içi çakışma artar. Özelleştirme sıralamayı ortadan kaldırmaz ama onu çip üstüne taşır (paylaşımlı bellek atomikleri ~10× hızlı) ve global atomikleri blok başına 256'ya indirir. Koşularda birleştirme (aggregation) tek atomikle k sayımı yazar.\")\n    R_.test(\"atomik çakışma\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: (max_loops(src, 0, func=\"reduce_by_key\"), max_loops(src, 0, func=\"segmented_max\")))\n\n    def t():\n        rng = np.random.default_rng(0)\n        k = rng.integers(0, 50, 10000)\n        v = rng.normal(size=10000)\n        uk, s = need(ns, \"reduce_by_key\")(k, v)\n        close(uk, np.unique(k), what=\"anahtarlar\")\n        close(s, np.array([v[k == a].sum() for a in np.unique(k)]), what=\"toplamlar\")\n        flags = np.zeros(20, int)\n        flags[[0, 3, 4, 11]] = 1\n        x = rng.normal(size=20)\n        close(need(ns, \"segmented_max\")(x, flags), np.array([x[0:3].max(), x[3:4].max(), x[4:11].max(), x[11:].max()]), what=\"segment max\")\n        R_.note(\"Anahtara göre indirgeme = sırala + segmentli indirgeme: veritabanlarında GROUP BY, grafiklerde düğüm başına toplama (scatter-add, GNN), seyrek matris-vektör çarpımında satır toplamları. GPU'da segment sınırları bir taramayla (M10.6) bulunur.\")\n    R_.test(\"segmentli indirgeme\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_5_reduction as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**İndirgeme:** işleç ✦ (toplam, çarpım, min, max, …) ve birim eleman (0, 1, +∞, −∞) ile bir diziyi tek değere "özetle". Sıralı: `acc = birim; for x: acc = acc ✦ x` → N adım. **Ağaç:** 1. turda N/2 çift paralel, 2. turda N/4, … → $\log_2N$ adım; toplam iş $N/2+N/4+\dots+1=N-1$ (sıralıyla aynı: iş verimli). Yeterli kaynak varsa hızlanma N/log N.

**CUDA'da blok içi ağaç** (her thread iki eleman, paylaşımlı dizi):

- **Basit eşleme:** `i = 2*t; for (s = 1; s <= blockDim; s *= 2) { if (t % s == 0) in[i] += in[i + s]; __syncthreads(); }` — sahip konumlar 2s'nin katları; ikinci iterasyondan itibaren **her warp ayrışır**, son iterasyonlarda 31/32 kaynak boşa.
- **Yakınsak eşleme:** `i = t; for (s = blockDim; s >= 1; s /= 2) { if (t < s) in[i] += in[i + s]; __syncthreads(); }` — aktif thread'ler bitişik, warp'lar ya tamamen aktif ya pasif; ayrıca ardışık thread'ler ardışık adreslere erişir (birleştirilmiş, bank çakışmasız).
- **Paylaşımlı bellek:** ilk toplamayı global okumada yap, ara sonuçlar çip üstünde.
- **Hiyerarşi:** her blok kendi parçasını indirger, sonucu `atomicAdd` ile globale ekler (ya da ikinci çekirdek).
- **Kabalaştırma:** her thread önce C elemanı sıralı toplar → daha az blok, daha az senkronizasyon (iş aynı, yol biraz uzar — donanım zaten yetersizse bedava).
- **Warp karıştırma:** son 32 eleman paylaşımlı bellek olmadan `__shfl_xor_sync` kelebeğiyle.

**Histogram:** her girdi bir kutuya +1 — çıkış konumu veriye bağlı (dağıtım/scatter) → yarış. Global `atomicAdd`: aynı kutuya giden thread'ler sıralanır (tek renkli görüntüde felaket). **Özelleştirme:** blok başına paylaşımlı bellekte alt-histogram, sonunda globale birleştir; **birleştirme** (aggregation): ardışık aynı değerleri sayıp tek atomik.

**Kayan nokta birleşmeli değildir:** sıralı toplam O(Nε) hata (float32'de 2²⁴'ten sonra +1 kaybolur), ağaç (ikili) O(ε log N), Kahan telafili toplam O(ε). Paralel indirgemenin sonucu sıralıdan farklı — çoğu zaman daha doğru.

## 1 · Elle

<!-- çözüm -->

Max ağacı: (3,1,7,0,4,1,6,3) → (3,7,4,6) → (7,6) → 7. Basit eşleme: adım 1…32'de 8 warp da aktif (6 iterasyon: 48), sonra 4, 2, 1 → 55. Yakınsak: 8, 4, 2, 1, sonra 5 iterasyon tek warp → 20 (ve bunlardan yalnız son 5'i ayrışık).

Float32: $2^{24}+1$ temsil edilemez (24 bitlik anlam); $2^{24}$ olduktan sonra her +1 en yakına yuvarlanır ve kaybolur.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**[3, 1, 7, 0, 4, 1, 6, 3] max-indirgemesi**

- `res`: Sonuç
- `steps`: Ağaç adımı
- `ops`: Toplam max işlemi

**256 thread'lik blok, 512 eleman (8 warp): tüm iterasyonlarda etkin warp toplamı**

- `ws`: Basit eşleme (adım 1, 2, …, 256; t mod adım = 0)
- `wc`: Yakınsak eşleme (adım 256, 128, …, 1; t < adım)

**Hiyerarşi ve hassasiyet**

- `bl`: 2²⁰ eleman, blok 1024 thread × 2 eleman: blok sayısı
- `blc`: Kabalaştırma çarpanı 8 ile blok sayısı
- `f32`: 10⁸ adet 1.0'ın float32'de sıralı toplamı

In [ ]:
cevaplar = dict(
    res=None,
    steps=None,
    ops=None,
    ws=None,
    wc=None,
    bl=None,
    blc=None,
    f32=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"res":[7,0],"steps":[3,0],"ops":[7,0],"ws":[55,0],"wc":[20,0],"bl":[512,0],"blc":[64,0],"f32":[16777216,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `reduce_tree(xs, op, identity)` → `(sonuç, adım, işlem)`: her turda çiftleri birleştir (tek sayıda ise son elemanı birim ile eşle ama sayma).
- `warp_activity(n_threads, mapping)` → her iterasyon için `(aktif thread, aktif warp, ayrışan warp)` listesi; `mapping` `"simple"` ya da `"convergent"`; 2·n_threads eleman.
- `run_block(x, mapping)` — aynı iki eşlemeyi gerçekten `x` (uzunluk 2·blockDim) üzerinde çalıştır (yerinde toplama), sonucu döndür.
- `hist_private(data, n_threads, n_bins)` → `(histogram, birleştirme_toplamaları)`: her "thread"in kendi kopyası (veri döngüsel dağıtılır), sonra birleştirme.

In [ ]:
%%writefile stage_b.py
def reduce_tree(xs, op, identity):
    ...


def warp_activity(n_threads, mapping):
    ...


def run_block(x, mapping):
    ...


def hist_private(data, n_threads, n_bins):
    ...


print(reduce_tree([3, 1, 7, 0, 4, 1, 6, 3], max, float("-inf")))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; döngü yalnız ağaç seviyeleri üzerinde.

- `tree_reduce(x, op, identity)` — `op` bir ufunc (`np.add`, `np.maximum`, …); her seviyede `op(x[0::2], x[1::2])`, tek uzunlukta birim ile doldur.
- `naive_sum32(x)` (float32 sıralı: `np.cumsum(dtype=float32)[-1]`) ve `tree_sum32(x)` (float32 ağaç).
- `argmax_tree(x)` — (değer, indeks) çiftlerinin ağaç indirgemesi; eşitlikte küçük indeks.
- `chain_product_tree(M)` — `M (n, 2, 2)` matrisler, sırayı koruyan ağaç çarpımı (`np.einsum`/`@`).

In [ ]:
%%writefile stage_c.py
import numpy as np


def tree_reduce(x, op, identity):
    ...


def naive_sum32(x):
    ...


def tree_sum32(x):
    ...


def argmax_tree(x):
    ...


def chain_product_tree(M):
    ...


print(tree_reduce(np.array([3, 1, 7, 0, 4, 1, 6, 3]), np.maximum, -np.inf))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Triton (GPU yoksa yorumlayıcı).

- `block_sums(x, BLOCK=1024)` — her program bloğunu `tl.sum` ile indirger → kısmi toplamlar.
- `sum_two_pass(x)` — kısmi toplamlara yeniden uygula (tek blok kalana dek).
- `sum_atomic(x)` — her program kısmi toplamını `tl.atomic_add` ile tek hücreye ekler.
- `argmax(x)` — blok başına (max, indeks), sonra torch ile son indirgeme.
- `histogram(x, nbins)` — int32 girdiler, `tl.atomic_add(H + v, 1)`.
- `butterfly(x)` (PyTorch, Triton değil) — `x (n_warps, 32)`; `__shfl_xor_sync` taklidi: `for off in 16, 8, 4, 2, 1: x = x + x[:, lane ^ off]`; her şerit warp toplamını tutar.

In [ ]:
%%writefile stage_d.py
import os
import torch

if not torch.cuda.is_available():
    os.environ.setdefault("TRITON_INTERPRET", "1")
import triton
import triton.language as tl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def block_sums(x, BLOCK=1024):
    ...


def sum_two_pass(x):
    ...


def sum_atomic(x):
    ...


def argmax(x):
    ...


def histogram(x, nbins):
    ...


def butterfly(x):
    ...


print(butterfly(torch.arange(64.0).reshape(2, 32))[:, :3])

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

C++ (`std::thread`) — CPU'da indirgeme ve histogramın ince noktaları.

- `sum_acc(x, nthreads, k)` — float32 dizi, thread başına **k ayrı biriktirici** (döngüyü k'lık açıp `acc[j] += x[i + j]`), sonunda birleştir; double döndür. k = 1'de her toplama bir öncekini bekler (gecikme zinciri), k ≥ 4 komut düzeyi paralellik (ILP) sağlar.
- `hist_copies(img, nthreads, copies)` — thread başına `copies` adet alt-histogram (piksel i → kopya i mod copies); tek renkli görüntüde aynı kutuya art arda yazma zinciri kırılır.

In [ ]:
%%writefile stage_e.py
import hashlib
import torch
from torch.utils.cpp_extension import load_inline

SRC = r"""
#include <torch/extension.h>
#include <thread>
#include <vector>

double sum_acc(torch::Tensor x, int64_t nthreads, int64_t k) {
    // TODO
    return 0.0;
}

torch::Tensor hist_copies(torch::Tensor img, int64_t nthreads, int64_t copies) {
    // TODO
    return torch::zeros({256}, torch::kInt64);
}
"""


def build():
    name = "ptl_m105e_" + hashlib.md5(SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=SRC, functions=["sum_acc", "hist_copies"], extra_cflags=["-O2"], verbose=False)


ext = build()
print(ext.sum_acc(torch.ones(1000), 2, 4))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Kayan nokta toplamı: sıralı, ikili, Kahan

- `kahan_sum(x)` — float32 telafili toplam (NumPy skalerleriyle, döngü serbest).
- `errors(n, seed=0)` → `(sıralı, ikili, kahan, np.sum)` göreli hataları: `x = rng.random(n).astype(float32)`, referans float64 `math.fsum`.

In [ ]:
%%writefile extra_x1.py
import math
import numpy as np


def kahan_sum(x):
    ...


def errors(n, seed=0):
    ...


print(errors(100000))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Histogramda atomik çakışma modeli

Global atomik çakışma modeli: bir warp'ın 32 thread'i atomikleri aynı anda yapar; aynı kutuya düşen k thread k adımda sıralanır → warp'ın maliyeti = en kalabalık kutu.

- `warp_serialization(bins)` — `bins (n_warps, 32)` → warp başına en büyük kutu tekrarı (NumPy, döngüsüz).
- `cost(image, n_bins=256, private=False, block=256)` → global atomik "adımlarının" toplamı; `private=True`: blok içi atomikler (aynı model) + blok başına n_bins global atomik.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def warp_serialization(bins):
    ...


def cost(image, n_bins=256, private=False, block=256):
    ...


print(cost(np.zeros(1024, np.int64)), cost(np.arange(1024) % 256))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Anahtara göre indirgeme (segmentli toplam)

- `reduce_by_key(keys, vals)` → `(benzersiz anahtarlar, toplamlar)`: sırala, segment başlarını bul, `np.add.reduceat` (döngüsüz).
- `segmented_max(vals, flags)` — `flags` segment başlarında 1; segment başına max (`np.maximum.reduceat`).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def reduce_by_key(keys, vals):
    ...


def segmented_max(vals, flags):
    ...


print(reduce_by_key(np.array([3, 1, 3, 2, 1]), np.array([1.0, 2, 3, 4, 5])))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Harris: CUDA'da paralel indirgemeyi optimize etmek

[Harris — Optimizing Parallel Reduction in CUDA (NVIDIA)](https://developer.download.nvidia.com/assets/cuda/files/reduction.pdf)

Sunumdaki yedi çekirdeği (aralıklı adresleme + ayrışan dal → bank çakışmalı aralıklı → sıralı adresleme → yüklemede ilk toplama → son warp açma → tam açma (şablon) → thread başına çok eleman) CUDA C++ ile yaz; 2²² int için GB/s ve her adımın hızlanmasını tabloya dök, b aşamasındaki warp sayımları ve M10.4'teki bank modeliyle açıkla. Son olarak `__shfl_down_sync` ile warp içi indirgemeyi ekle ve CUB `DeviceReduce::Sum` ile karşılaştır.


**Adımlar**

1. Son warp açmada `volatile` paylaşımlı işaretçinin neden gerektiğini (ve Volta sonrası neden `__syncwarp` gerektiğini) açıkla.

**Kabul ölçütleri**

- 7. çekirdek tepe bant genişliğinin ≥ %70'i.
- Her optimizasyonun etkisi tablo ve kısa açıklamayla.

## Makale ödevi: Podlozhnyuk (2007): CUDA'da histogram

[Podlozhnyuk — Histogram calculation in CUDA (NVIDIA)](https://developer.download.nvidia.com/compute/cuda/1.1-Beta/x86_website/projects/histogram64/doc/histogram.pdf)

lena (ya da kendi) gri görüntün için 256 kutulu histogramı (a) global atomiklerle, (b) blok başına paylaşımlı bellek alt-histogramıyla, (c) warp başına alt-histogram + kabalaştırma ile yaz; tekdüze, doğal ve tek renkli (tüm pikseller aynı) görüntülerde zamanla. x2'deki çakışma modelinin öngörüsüyle karşılaştır; M10.3'teki çok thread'li C++ sürümüyle (Ex. 3) hızlanmayı raporla.


**Adımlar**

1. Kopya → hesap → geri kopya sürelerini ayrı ölç (cudaEvent).

**Kabul ölçütleri**

- Tek renkli görüntüde (a) çok yavaşlar, (c) sağlam kalır.

## Kaynaklar

- [Harris — Optimizing Parallel Reduction in CUDA (NVIDIA)](https://developer.download.nvidia.com/assets/cuda/files/reduction.pdf) — ©
- [Hwu, Kirk, El Hajj — Programming Massively Parallel Processors (4. baskı), Bölüm 9–10](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — ©
- [Podlozhnyuk — Histogram calculation in CUDA (NVIDIA 2007)](https://developer.download.nvidia.com/compute/cuda/1.1-Beta/x86_website/projects/histogram64/doc/histogram.pdf) — ©
- [Higham — The Accuracy of Floating Point Summation (SIAM J. Sci. Comput. 1993)](https://doi.org/10.1137/0914050) — ©
- [NVIDIA — Using CUDA Warp-Level Primitives (Developer Blog)](https://developer.nvidia.com/blog/using-cuda-warp-level-primitives/) — ©